<a name="top"></a><img src="../images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 2.4：时序逻辑
**上一节：[控制流](2.3_control_flow.ipynb)**<br>
**下一节：[FIR 滤波器](2.5_exercise.ipynb)**

## 动机
没有状态，你就无法编写任何有意义的数字逻辑。没有状态，你就无法编写任何有意义的数字逻辑。没有状态，你就无法编写任何有意义的数字逻辑……

明白了吗？因为如果不存储中间结果，你哪儿也去不了。

好吧，抛开这个烂笑话不谈，本模块将描述如何在 Chisel 中表达常见的时序逻辑模式。学完本模块后，你应该能够在 Chisel 中实现并测试一个移位寄存器。

需要强调的是，本节可能不会给你带来极大的震撼。Chisel 的优势不在于新的时序逻辑模式，而在于设计的参数化。在我们展示这种能力之前，必须先了解这些时序模式是什么。因此，本节将向你展示 Chisel 几乎可以做到 Verilog 能做的所有事情——你只需要学习 Chisel 的语法。

## 环境设置

In [ ]:
val path = System.getProperty("user.dir") + "/../source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.tester._
import chisel3.tester.RawTester.test

---
# 寄存器
Chisel 中基本的带状态元件是寄存器，用 `Reg` 表示。
一个 `Reg` 保持其输出值，直到时钟上升沿，此时它会接收其输入的值。
默认情况下，每个 Chisel `Module` 都有一个隐式时钟，设计中的每个寄存器都使用该时钟。
这使你不必在代码中反复指定同一个时钟。

<span style="color:blue">**示例：使用寄存器**</span><br>
以下代码块实现了一个模块，该模块接收输入、加 1，然后将其连接为寄存器的输入。
*注意：对于多时钟设计，隐式时钟可以被覆盖。请参阅附录中的示例。*

In [ ]:
class RegisterModule extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(12.W))
    val out = Output(UInt(12.W))
  })
  
  val register = Reg(UInt(12.W))
  register := io.in + 1.U
  io.out := register
}

test(new RegisterModule) { c =>
  for (i <- 0 until 100) {
    c.io.in.poke(i.U)
    c.clock.step(1)
    c.io.out.expect((i + 1).U)
  }
}
println("SUCCESS!!")

寄存器是通过调用 `Reg(tpe)` 创建的，其中 `tpe` 是一个编码了所需寄存器类型的变量。
在这个例子中，`tpe` 是一个 12 位的 `UInt`。

看看上面的测试器在做什么。
在 `poke()` 和 `expect()` 的调用之间，有一个 `step(1)` 的调用。
这告诉测试框架将时钟向前推进一个周期，这将导致寄存器将其输入传递到其输出。

调用 `step(n)` 会将时钟向前推进 `n` 个周期。

细心的观察者会注意到，之前测试组合逻辑的测试器并没有调用 `step()`。这是因为在输入上调用 `poke()` 会立即将更新后的值传播到整个组合逻辑。只有在更新时序逻辑中的状态元件时才需要调用 `step()`。

下面的代码块将展示 `RegisterModule` 生成的 Verilog。

注意：
* 模块有一个你没有添加的时钟（和复位）输入——这就是隐式时钟
* 变量 `register` 如预期那样显示为 `reg [11:0]`
* 有一个由 `ifdef Randomize` 分隔的代码块，用于在仿真开始前将寄存器初始化为某个随机变量
* `register` 在 `posedge clock` 上更新

In [ ]:
println(getVerilog(new RegisterModule))

一个重要的问题是，Chisel 会区分类型（如 `UInt`）和硬件节点（如字面量 `2.U`，或 `myReg` 的输出）。虽然
```scala
val myReg = Reg(UInt(2.W))
```
是合法的，因为 Reg 需要一种数据类型作为模板，
```scala
val myReg = Reg(2.U)
```
是一个错误，因为 `2.U` 已经是一个硬件节点，不能用作模板。

<span style="color:blue">**示例：RegNext**</span><br>
Chisel 有一个便捷的寄存器对象，用于输入连接简单的寄存器。之前的 `Module` 可以简化为下面的 `Module`。请注意，这次我们不需要指定寄存器的位宽。它可以从寄存器的输出连接推断出来，在这个例子中，就是 `io.out`。

In [ ]:
class RegNextModule extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(12.W))
    val out = Output(UInt(12.W))
  })
  
  // 寄存器位宽从 io.out 推断
  io.out := RegNext(io.in + 1.U)
}

test(new RegNextModule) { c =>
  for (i <- 0 until 100) {
    c.io.in.poke(i.U)
    c.clock.step(1)
    c.io.out.expect((i + 1).U)
  }
}
println("SUCCESS!!")

Verilog 看起来与之前几乎相同，只是寄存器名称是由生成的，而不是显式定义的。

In [ ]:
println(getVerilog(new RegNextModule))

---
# `RegInit`

`RegisterModule` 中的寄存器在仿真中被初始化为随机数据。
除非另有指定，否则寄存器没有复位值（也没有复位）。
创建一个复位到给定值的寄存器的方法是使用 `RegInit`。

例如，一个初始化为零的 12 位寄存器可以通过以下方式创建。
以下两种写法都是有效的，并且功能相同：
```scala
val myReg = RegInit(UInt(12.W), 0.U)
val myReg = RegInit(0.U(12.W))
```

第一种写法有两个参数。
第一个参数是一个类型节点，指定了数据类型及其位宽。
第二个参数是一个硬件节点，指定了复位值，这里为 0。

第二种写法有一个参数。
它是一个硬件节点，指定了复位值，但通常用 `0.U`。

<span style="color:blue">**示例：初始化寄存器** </span><br>
以下演示了使用 `RegInit()`，初始化为零。

In [ ]:
class RegInitModule extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(12.W))
    val out = Output(UInt(12.W))
  })
  
  val register = RegInit(0.U(12.W))
  register := io.in + 1.U
  io.out := register
}

println(getVerilog(new RegInitModule))

请注意，生成的 Verilog 现在有一个检查 `if (reset)` 的代码块，用于将寄存器复位为 0。
另外请注意，这是在 `always @(posedge clock)` 代码块内部的。
Chisel 的隐式复位是高电平有效且同步的。
在调用复位之前，寄存器仍然被初始化为随机垃圾数据。
`PeekPokeTesters` 总是在运行测试之前调用复位，但你也可以使用 `reset(n)` 函数手动调用复位，其中复位信号会保持高电平 `n` 个周期。

---
# 控制流
寄存器在控制流方面与线网非常相似。
它们具有最后连接语义，并且可以使用 `when`、`elsewhen` 和 `otherwise` 进行条件赋值。

<span style="color:blue">**示例：寄存器控制流**</span><br>
以下示例使用条件寄存器赋值来找出输入序列中的最大值。

In [ ]:
class FindMax extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(10.W))
    val max = Output(UInt(10.W))
  })

  val max = RegInit(0.U(10.W))
  when (io.in > max) {
    max := io.in
  }
  io.max := max
}

test(new FindMax) { c =>
    c.io.max.expect(0.U)
    c.io.in.poke(1.U)
    c.clock.step(1)
    c.io.max.expect(1.U)
    c.io.in.poke(3.U)
    c.clock.step(1)
    c.io.max.expect(3.U)
    c.io.in.poke(2.U)
    c.clock.step(1)
    c.io.max.expect(3.U)
    c.io.in.poke(24.U)
    c.clock.step(1)
    c.io.max.expect(24.U)
}
println("SUCCESS!!")

---
# 其他寄存器示例

在寄存器上调用的操作是在寄存器的**输出**上执行的，操作的类型取决于寄存器的类型。
这意味着你可以这样写
```scala
val reg: UInt = Reg(UInt(4.W))
```
这意味着值 `reg` 的类型是 `UInt`，你可以对它执行通常可以对 `UInt` 执行的操作，比如 `+`、`-` 等。

你不仅限于在寄存器中使用 `UInt`，你可以使用基本类型 `chisel3.Data` 的任何子类。这包括用于有符号整数的 `SInt` 以及许多其他类型。

<span style="color:blue">**示例：梳状滤波器**</span><br>
以下示例展示了一个梳状滤波器。

In [ ]:
class Comb extends Module {
  val io = IO(new Bundle {
    val in  = Input(SInt(12.W))
    val out = Output(SInt(12.W))
  })

  val delay: SInt = Reg(SInt(12.W))
  delay := io.in
  io.out := io.in - delay
}
println(getVerilog(new Comb))

---
# 练习
<span style="color:red">**练习：移位寄存器**</span><br>
运用你新学到的寄存器知识，构建一个实现 LFSR（线性反馈移位寄存器）中移位寄存器的模块。具体要求如下：
- 每个元素为 1 位宽。
- 有一个 4 位输出信号。
- 接收一个单输入位，即移位寄存器的下一个输入值。
- 输出移位寄存器的并行输出，最高有效位（MSB）是移位寄存器的最后一个元素，最低有效位（LSB）是移位寄存器的第一个元素。`Cat` 可能会派上用场。
- **输出初始化为 `b0001`。**
- 每个时钟周期移位一次（无使能信号）。
- **请注意，在 Chisel 中，子字赋值是非法的**；像 `out(0) := in` 这样的写法是行不通的。

<img src="images/shifter4.svg" alt="shift register figure" style="width: 450px" />

下面提供了一个基本的模块骨架、测试向量和驱动调用。第一个寄存器已经为你提供。

In [ ]:
class MyShiftRegister(val init: Int = 1) extends Module {
  val io = IO(new Bundle {
    val in  = Input(Bool())
    val out = Output(UInt(4.W))
  })

  val state = RegInit(UInt(4.W), init.U)

  ???
}

test(new MyShiftRegister()) { c =>
  var state = c.init
  for (i <- 0 until 10) {
    // poke in LSB of i (i % 2)
    c.io.in.poke(((i % 2) != 0).B)
    // update expected state
    state = ((state * 2) + (i % 2)) & 0xf
    c.clock.step(1)
    c.io.out.expect(state.U)
  }
}
println("SUCCESS!!")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>解答</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  val nextState = (state << 1) | io.in
  state := nextState
  io.out := state
</pre></article></div></section></div>

<span style="color:red">**练习：参数化移位寄存器（可选）**</span><br>
编写一个移位寄存器，该寄存器通过其延迟（`n`）、其初始值（`init`）进行参数化，并且还有一个使能输入信号（`en`）。

In [ ]:
// n 是输出宽度（延迟数 - 1）
// 初始化状态为 init
class MyOptionalShiftRegister(val n: Int, val init: BigInt = 1) extends Module {
  val io = IO(new Bundle {
    val en  = Input(Bool())
    val in  = Input(Bool())
    val out = Output(UInt(n.W))
  })

  val state = RegInit(init.U(n.W))

  ???
}

// 测试不同的深度
for (i <- Seq(3, 4, 8, 24, 65)) {
  println(s"Testing n=$i")
  test(new MyOptionalShiftRegister(n = i)) { c =>
    val inSeq = Seq(0, 1, 1, 1, 0, 1, 1, 0, 0, 1)
    var state = c.init
    var i = 0
    c.io.en.poke(true.B)
    while (i < 10 * c.n) {
      // poke in repeated inSeq
      val toPoke = inSeq(i % inSeq.length)
      c.io.in.poke((toPoke != 0).B)
      // update expected state
      state = ((state * 2) + toPoke) & BigInt("1"*c.n, 2)
      c.clock.step(1)
      c.io.out.expect(state.U)

      i += 1
    }
  }
}
println("SUCCESS!!")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-2" />
<label for="check-2"><strong>解答</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  val nextState = (state << 1) | io.in
  when (io.en) {
    state  := nextState
  }
  io.out := state
</pre></article></div></section></div>

---
# 附录：显式时钟和复位
Chisel 模块有一个默认的时钟和复位，它们被模块内部创建的每个寄存器隐式使用。
有时你可能希望覆盖这种默认行为；也许你有一个生成时钟或复位信号的黑盒，或者你有一个多时钟设计。

Chisel 提供了处理这些情况的构造。
时钟和复位可以单独覆盖，也可以使用 `withClock() {}`、`withReset() {}` 和 `withClockAndReset() {}` 一起覆盖。
以下代码块将给出使用这些函数的示例。

需要注意的一点是，`reset`（在编写本教程时）始终是同步的，并且类型为 `Bool`。
时钟在 Chisel 中有自己的类型（`Clock`），应如此声明。
*`Bool` 可以通过调用 `asClock()` 转换为 `Clock`，但你应该小心，确保没有做傻事。*

另请注意，`chisel-testers` 目前尚未完全支持多时钟设计。

<span style="color:blue">**示例：多时钟模块**</span><br>
一个具有多个时钟和复位信号的模块。

In [ ]:
// 我们需要导入多时钟特性
import chisel3.experimental.{withClock, withReset, withClockAndReset}

class ClockExamples extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(10.W))
    val alternateReset    = Input(Bool())
    val alternateClock    = Input(Clock())
    val outImplicit       = Output(UInt())
    val outAlternateReset = Output(UInt())
    val outAlternateClock = Output(UInt())
    val outAlternateBoth  = Output(UInt())
  })

  val imp = RegInit(0.U(10.W))
  imp := io.in
  io.outImplicit := imp

  withReset(io.alternateReset) {
    // 此作用域中的所有内容都将使用 alternateReset 作为复位
    val altRst = RegInit(0.U(10.W))
    altRst := io.in
    io.outAlternateReset := altRst
  }

  withClock(io.alternateClock) {
    val altClk = RegInit(0.U(10.W))
    altClk := io.in
    io.outAlternateClock := altClk
  }

  withClockAndReset(io.alternateClock, io.alternateReset) {
    val alt = RegInit(0.U(10.W))
    alt := io.in
    io.outAlternateBoth := alt
  }
}

println(getVerilog(new ClockExamples))

---
# 总结
出色地完成了本节！！你现在已经学会了如何在 Chisel 中创建寄存器和编写时序逻辑，这意味着你已经掌握了足够的基本构建块来编写真正的电路。

下一节将把我们学到的一切整合到一个示例中！如果你需要一点额外的鼓励，请记住一位 Chisel 专家用户的这句话：

![BobRoss](http://i.qkme.me/3qbd5u.jpg)

---
# 你已完成！

[返回顶部。](#top)